In [2]:
!nvidia-smi -L
!nvidia-smi

GPU 0: Tesla T4 (UUID: GPU-804ac4e5-17a3-94af-044c-ed6f3264755f)
Tue Jun 24 08:54:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |       

 Environment Setup

In [3]:
!pip install transformers accelerate scipy soundfile torch torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [42]:
import torch
import soundfile as sf
from transformers import BarkModel, BarkProcessor, AutoProcessor
import IPython.display as ipd

In [49]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
model = BarkModel.from_pretrained("suno/bark").to(device) # Using bark-small for faster loading/inference
processor = BarkProcessor.from_pretrained("suno/bark")
processor_1 = AutoProcessor.from_pretrained("suno/bark")

Using device: cuda


In [51]:
# Define your text input
text_prompt = """Speech synthesis is the artificial production of human speech. A computer system used for this purpose is called a speech synthesizer, and can be implemented in software or hardware products."""
# Preprocess the text and move to the correct device
inputs = processor(text_prompt, return_tensors="pt").to(device)

# Generate speech
# You can adjust `do_sample=True` and `temperature` for more varied outputs.
# `fine_temperature` can also influence expressiveness.
speech_output = model.generate(**inputs, do_sample=True, fine_temperature=0.7, top_k=50, top_p=0.9)

# Convert the output to a NumPy array and play it
sampling_rate = model.generation_config.sample_rate
audio_array = speech_output.cpu().numpy().squeeze()

print("Playing generated audio...")
ipd.display(ipd.Audio(audio_array, rate=sampling_rate))

# Optionally, save the audio to a WAV file
output_filename = "bark_speech_output.wav"
sf.write(output_filename, audio_array, sampling_rate)
print(f"Audio saved to {output_filename}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.


Playing generated audio...


Audio saved to bark_speech_output.wav


In [ ]:

# Input text (long)
text_prompt = """
It started on a rainy Thursday, the kind that turns city streets into silver rivers. Maya stood under a crooked awning, her umbrella long defeated by the wind. That's when she saw him—Alex—offering his own umbrella without a word, just a gentle smile.
They walked together in silence, footsteps syncing as if they'd known each other for years. The city noise faded behind the sound of soft laughter and shared warmth.
Over coffee that smelled like cinnamon and second chances, they talked. About books. About fears. About the little things that made them feel alive. And when they laughed, it echoed like music neither of them realized they’d missed.
Days turned into weeks. Umbrellas turned into excuses to meet. And kisses turned into promises spoken not with words, but with eyes that dared to dream again.
Then came the evening Maya cried—not from sadness, but from the overwhelming joy of being seen, truly seen, for the first time. Alex held her, saying nothing, because some silences say everything.
On the same corner where they first met, months later, the rain returned.
Alex took her hand. “Let’s never run from the rain again,” he whispered.
Maya smiled. “Only if you promise to get soaked with me.”
And so they stood, drenched in love, where it all began—with a broken umbrella and a beginning they never saw coming.

"""

# Helper: Split text into ~100-word chunks
def split_into_chunks(text, max_words=50):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]


# Process each chunk
chunks = split_into_chunks(text_prompt)
print(f"Processing {len(chunks)} chunks...")
print(f"{chunks}")

model.generation_config.history_prompt = "v2/en_speaker_6"

sampling_rate = model.generation_config.sample_rate
full_audio = []

for i, chunk in enumerate(chunks):
    print(f"Generating chunk {i+1}/{len(chunks)}...{chunk}")
    inputs = processor(chunk, return_tensors="pt").to(device)
    speech = model.generate(**inputs)
    audio = speech.cpu().numpy().squeeze()
    full_audio.append(audio)


# Concatenate all audio parts
final_audio = np.concatenate(full_audio)

# Playback
print("Playing full audio...")
ipd.display(ipd.Audio(final_audio, rate=sampling_rate))

# Save to file
output_filename = "bark_long_output.wav"
sf.write(output_filename, final_audio, sampling_rate)
print(f"Saved to {output_filename}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.


Processing 5 chunks...
["It started on a rainy Thursday, the kind that turns city streets into silver rivers. Maya stood under a crooked awning, her umbrella long defeated by the wind. That's when she saw him—Alex—offering his own umbrella without a word, just a gentle smile. They walked together in silence, footsteps syncing", "as if they'd known each other for years. The city noise faded behind the sound of soft laughter and shared warmth. Over coffee that smelled like cinnamon and second chances, they talked. About books. About fears. About the little things that made them feel alive. And when they laughed, it", 'echoed like music neither of them realized they’d missed. Days turned into weeks. Umbrellas turned into excuses to meet. And kisses turned into promises spoken not with words, but with eyes that dared to dream again. Then came the evening Maya cried—not from sadness, but from the overwhelming joy of', 'being seen, truly seen, for the first time. Alex held her, saying nothin

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.


Generating chunk 2/5...as if they'd known each other for years. The city noise faded behind the sound of soft laughter and shared warmth. Over coffee that smelled like cinnamon and second chances, they talked. About books. About fears. About the little things that made them feel alive. And when they laughed, it
